# SoccerNet Goal Spotting with R2Plus1D

**Pipeline overview**
1. Load SoccerNet official train/valid/test splits
2. Parse `Labels-v2.json` annotations to extract goal timestamps per half
3. Build a `ClipDataset` that samples positive clips (near goals) and negative clips (background)
4. Train `R2Plus1D-18` as a binary classifier (goal vs. background)
5. Sliding-window inference over every game half to produce a confidence curve
6. Non-maximum suppression to get predicted timestamps
7. Evaluate with SoccerNet-style Average-mAP at multiple tolerances

**Why clip-level classification rather than direct timestamp regression?**
R2Plus1D was designed for fixed-length clip classification. The SoccerNet action spotting
benchmark uses this paradigm: train a per-clip binary classifier, slide the window over the
full half, and pick the timestamp at the peak of the confidence curve. This is equivalent to
timestamp prediction but far simpler to train than an anchor-free temporal detector.

In [1]:
# 0. Config
import os

PROJECT_DIR   = "/home/jinny/aspotting"
DATA_DIR      = f"{PROJECT_DIR}/dataset"
CKPT_DIR      = f"{PROJECT_DIR}/checkpoints/e2"
RESUME_FROM   = ""   # path to a latest.pt to resume from, or "" for fresh start
MANIFEST_DIR  = f"{PROJECT_DIR}/manifests/e2"
CLIP_CACHE_DIR = f"{PROJECT_DIR}/clip_cache/e2_center"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(MANIFEST_DIR, exist_ok=True)
os.makedirs(CLIP_CACHE_DIR, exist_ok=True)

FPS           = 25
CLIP_SEC      = 4.0
CLIP_FRAMES   = 16
CLIP_SIZE     = (112, 112)

BATCH_SIZE    = 16
EPOCHS        = 8
LR            = 1e-4
WEIGHT_DECAY  = 1e-3

# Negative sampling
HARD_NEG_LABELS = {
    "shots on target",
    "shots off target",
    "penalty",
}
USE_HARD_NEGS        = True
NEG_HARD_PER_GOAL    = 1
NEG_RAND_PER_GOAL    = 4
NEG_PER_NO_GOAL_HALF = 12
MIN_NEG_FROM_GOAL_SEC = 12.0

# Manifest
FORCE_REBUILD_MANIFEST = True  # set True to regenerate from scratch
USE_JITTER             = False   # set False for center-only clips (pair with e2_center cache)

# Clip cache
# Run cell 5b once to extract clips, then set USE_CLIP_CACHE=True for faster training
USE_CLIP_CACHE = True

# Augmentation
# Applies random horizontal flip during training only. Val/inference clips are never augmented.
USE_AUGMENTATION = True

# Model
FREEZE_BACKBONE = False  # freeze stem + layer1-3, only train layer4 + fc
DROPOUT_P       = 0.4    # dropout before the FC head (set 0.0 to disable)

STRIDE_S      = 2.0
NMS_RADIUS_S  = 10.0
CONF_THRESH   = 0.5

TOLERANCES    = [5, 10, 30, 60]

DEVICE = "cuda" if __import__('torch').cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Hard negatives: {USE_HARD_NEGS}  |  Clip cache: {USE_CLIP_CACHE}  |  Augmentation: {USE_AUGMENTATION}")
print(f"Freeze backbone: {FREEZE_BACKBONE}  |  Dropout: {DROPOUT_P}")
print(f"Jitter: {USE_JITTER}  |  Resume from: {RESUME_FROM or '(none)'}")

Device: cuda
Hard negatives: True  |  Clip cache: True  |  Augmentation: True
Freeze backbone: False  |  Dropout: 0.4
Jitter: False  |  Resume from: (none)


In [2]:
# 1. Imports
import json
import random
import hashlib
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r2plus1d_18, R2Plus1D_18_Weights
from tqdm.auto import tqdm
from dotenv import load_dotenv
import wandb

from SoccerNet.Downloader import getListGames

load_dotenv(dotenv_path="/home/jinny/aspotting/.env")
print("Imports OK")


/home/jinny/aspotting/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK


In [3]:
# 2. SoccerNet official splits
# getListGames returns relative paths like
# 'england_epl/2014-2015/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley'

all_splits = {}
for split in ("train", "valid", "test"):
    games = getListGames(split)
    local = [g for g in games if Path(DATA_DIR, g, "Labels-v2.json").exists()]
    all_splits[split] = local
    print(f"{split:5s}  total={len(games):3d}  local={len(local):3d}")

train  total=300  local=300
valid  total=100  local=100
test   total=100  local=100


In [4]:
# 3. Annotation parser
# parse_annotations returns goal times AND hard negative times per half.
# Hard negatives are shots, corners, penalties -- visually similar to goals.

def parse_annotations(game_rel_path):
    """
    Returns:
        goal_times : {half: [seconds, ...]}
        hard_times : {half: [(seconds, label), ...]}  — label is the specific event type
    """
    json_path = Path(DATA_DIR, game_rel_path, "Labels-v2.json")
    with open(json_path) as f:
        data = json.load(f)

    goal_times = defaultdict(list)
    hard_times = defaultdict(list)

    for ann in data["annotations"]:
        label = ann.get("label", "").strip().lower()
        half_str, _ = ann["gameTime"].split(" - ")
        half  = int(half_str)
        pos_s = int(ann["position"]) / 1000.0   # ms -> seconds

        if label == "goal":
            goal_times[half].append(pos_s)
        elif label in HARD_NEG_LABELS:
            hard_times[half].append((pos_s, label))   # store (time, label) tuple

    return dict(goal_times), dict(hard_times)


# Sanity check
sample_game = all_splits["train"][0]
g_times, h_times = parse_annotations(sample_game)
print(f"Sample game : {sample_game}")
print(f"Goal times  : {g_times}")
print(f"Hard neg times (half 1, first 3): {dict(list(h_times.items())[:1])}")

Sample game : england_epl/2014-2015/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley
Goal times  : {1: [790.722], 2: [2121.494]}
Hard neg times (half 1, first 3): {1: [(270.441, 'shots on target'), (347.458, 'shots on target'), (467.037, 'shots off target'), (866.879, 'shots on target'), (1033.041, 'shots off target'), (1323.355, 'shots on target'), (1440.175, 'shots off target')]}


In [5]:
# 4. Clip reader and Dataset

MEAN = np.array([0.43216, 0.394666, 0.37645],  dtype=np.float32)
STD  = np.array([0.22803, 0.22145,  0.216989], dtype=np.float32)


def read_clip(video_path, center_sec, n_frames=CLIP_FRAMES,
              clip_sec=CLIP_SEC, size=CLIP_SIZE):
    """
    Sample n_frames frames sparsely across clip_sec, centred on center_sec.
    Returns a (C, T, H, W) float32 normalised tensor, or None on failure.
    """
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None
    fps_v = cap.get(cv2.CAP_PROP_FPS) or FPS

    start_sec   = max(0.0, center_sec - clip_sec / 2)
    start_frame = int(start_sec * fps_v)
    step        = max(1, int(clip_sec * fps_v / n_frames))

    frames = []
    for i in range(n_frames):
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame + i * step)
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, size)
        frames.append(frame)
    cap.release()

    while len(frames) < n_frames:
        frames.append(frames[-1] if frames else np.zeros((*size, 3), np.uint8))

    arr = np.stack(frames).astype(np.float32) / 255.0
    arr = (arr - MEAN) / STD
    return torch.from_numpy(arr).permute(3, 0, 1, 2)    # (C, T, H, W)


def clip_cache_path(video_path, center_sec):
    """Returns the .npy path for a given clip, deterministic from its key."""
    key = f"{video_path}_{center_sec:.4f}"
    h   = hashlib.md5(key.encode()).hexdigest()
    return Path(CLIP_CACHE_DIR) / f"{h}.npy"


def far_from_all(t, times, min_gap):
    return all(abs(t - x) >= min_gap for x in times)


def build_sample_list(game_list, is_train=True):
    # Each sample is a 4-tuple: (video_path, center_sec, label, neg_type)
    # neg_type is "goal" for positives, the specific event label for hard negs,
    # and "rand" for random background negatives.
    samples = []
    for game in tqdm(game_list, desc="Building samples"):
        goal_times, hard_times = parse_annotations(game)
        for half in (1, 2):
            vid = Path(DATA_DIR, game, f"{half}_224p.mkv")
            if not vid.exists():
                continue
            cap   = cv2.VideoCapture(str(vid))
            fps_v = cap.get(cv2.CAP_PROP_FPS) or FPS
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            dur = total / fps_v
            if dur < CLIP_SEC + 1:
                continue

            lo    = CLIP_SEC / 2
            hi    = dur - CLIP_SEC / 2
            goals = goal_times.get(half, [])
            hard  = hard_times.get(half, [])   # list of (time, label) tuples

            jitters = (-2.0, 0.0, 2.0) if (is_train and USE_JITTER) else (0.0,)
            for g in goals:
                for j in jitters:
                    c = float(np.clip(g + j, lo, hi))
                    samples.append((str(vid), c, 1.0, "goal"))

            hard_cands = []
            if USE_HARD_NEGS:
                hard_cands = [(t, lbl) for t, lbl in hard
                              if far_from_all(t, goals, MIN_NEG_FROM_GOAL_SEC)]
                n_hard = min(len(hard_cands),
                             NEG_HARD_PER_GOAL * max(1, len(goals)))
                if n_hard > 0:
                    for t, lbl in random.sample(hard_cands, k=n_hard):
                        c = float(np.clip(t, lo, hi))
                        samples.append((str(vid), c, 0.0, lbl))

            n_rand    = (NEG_RAND_PER_GOAL * len(goals)
                         if goals else NEG_PER_NO_GOAL_HALF)
            forbidden = list(goals) + [t for t, _ in hard_cands]
            added = attempts = 0
            while added < n_rand and attempts < n_rand * 40:
                attempts += 1
                t = random.uniform(lo, hi)
                if far_from_all(t, forbidden, MIN_NEG_FROM_GOAL_SEC):
                    samples.append((str(vid), t, 0.0, "rand"))
                    forbidden.append(t)
                    added += 1

    return samples


class ClipDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        if USE_CLIP_CACHE:
            cache = clip_cache_path(row.video_path, float(row.center_sec))
            if cache.exists():
                clip = torch.from_numpy(
                    np.load(str(cache)).astype(np.float32))
            else:
                clip = read_clip(row.video_path, float(row.center_sec))
                if clip is None:
                    clip = torch.zeros(3, CLIP_FRAMES, *CLIP_SIZE)
        else:
            clip = read_clip(row.video_path, float(row.center_sec))
            if clip is None:
                clip = torch.zeros(3, CLIP_FRAMES, *CLIP_SIZE)
        if self.transform is not None:
            clip = self.transform(clip)
        return clip, torch.tensor(float(row.label), dtype=torch.float32)


print("Dataset utilities defined")

Dataset utilities defined


In [6]:
# 5. Build or load manifest, then create data loaders
# Set FORCE_REBUILD_MANIFEST=True in config to regenerate from scratch.
# Set it to False on subsequent runs to reuse the saved CSVs.

train_csv = f"{MANIFEST_DIR}/train_clips.csv"
valid_csv = f"{MANIFEST_DIR}/valid_clips.csv"
neg_label = "rand" if not USE_HARD_NEGS else "hard+rand"

if FORCE_REBUILD_MANIFEST or not (Path(train_csv).exists() and Path(valid_csv).exists()):
    random.seed(42)
    train_samples = build_sample_list(all_splits["train"], is_train=True)
    valid_samples = build_sample_list(all_splits["valid"], is_train=False)

    train_df = pd.DataFrame(train_samples, columns=["video_path", "center_sec", "label", "neg_type"])
    valid_df = pd.DataFrame(valid_samples, columns=["video_path", "center_sec", "label", "neg_type"])

    train_df.to_csv(train_csv, index=False)
    valid_df.to_csv(valid_csv, index=False)
    print(f"Manifests saved to {MANIFEST_DIR}")
else:
    train_df = pd.read_csv(train_csv)
    valid_df = pd.read_csv(valid_csv)
    print(f"Loaded manifests from {MANIFEST_DIR}")

n_pos = int((train_df.label == 1.0).sum())
n_neg = int((train_df.label == 0.0).sum())
print(f"Train: {len(train_df):,} clips  (pos={n_pos}, neg={n_neg})  neg_type={neg_label}")
print(f"  neg breakdown: {train_df[train_df.label==0.0]['neg_type'].value_counts().to_dict()}")
print(f"Valid: {len(valid_df):,} clips  "
      f"(pos={int((valid_df.label==1.0).sum())}, "
      f"neg={int((valid_df.label==0.0).sum())})")
print(f"  neg breakdown: {valid_df[valid_df.label==0.0]['neg_type'].value_counts().to_dict()}")

import torchvision.transforms.v2 as Tv2

# Applied to training clips only. Val/inference clips are never augmented.
_train_transform = Tv2.RandomHorizontalFlip(p=0.5) if USE_AUGMENTATION else None

train_loader = DataLoader(ClipDataset(train_df, transform=_train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
valid_loader = DataLoader(ClipDataset(valid_df, transform=None),
                          batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Train batches: {len(train_loader)}  Valid batches: {len(valid_loader)}")
print(f"Augmentation: {'ON (horizontal flip)' if USE_AUGMENTATION else 'OFF'}")

Building samples: 100%|██████████| 100/100 [00:00<00:00, 128.75it/s]

Manifests saved to /home/jinny/aspotting/manifests/e2
Train: 7,593 clips  (pos=995, neg=6598)  neg_type=hard+rand
  neg breakdown: {'rand': 5492, 'shots off target': 638, 'shots on target': 460, 'penalty': 8}
Valid: 2,741 clips  (pos=371, neg=2370)
  neg breakdown: {'rand': 1964, 'shots off target': 231, 'shots on target': 172, 'penalty': 3}
Train batches: 474  Valid batches: 172
Augmentation: ON (horizontal flip)


In [ ]:
# 5b. Extract and cache clips as .npy files
# Run this once after building the manifest.
# After it finishes set USE_CLIP_CACHE=True in config for faster training.
#
# Storage: clips saved as float16 -> ~15GB for 12,000 clips.
# Loading a .npy is a single sequential disk read with no video decoding,
# which is much faster than 16 random seeks into a compressed MKV.

def extract_and_cache_clips(df, desc="Extracting"):
    skipped = extracted = failed = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        out = clip_cache_path(row.video_path, float(row.center_sec))
        if out.exists():
            skipped += 1
            continue
        clip = read_clip(row.video_path, float(row.center_sec))
        if clip is None:
            failed += 1
            continue
        # Save as float16 to halve storage vs float32
        np.save(str(out), clip.numpy().astype(np.float16))
        extracted += 1
    print(f"  extracted={extracted}  skipped={skipped}  failed={failed}")

print(f"Cache dir : {CLIP_CACHE_DIR}")
print(f"Clips to extract: {len(train_df) + len(valid_df):,}")
print()

extract_and_cache_clips(train_df, desc="Train clips")
extract_and_cache_clips(valid_df, desc="Valid clips")

n_cached = len(list(Path(CLIP_CACHE_DIR).glob("*.npy")))
size_gb  = sum(f.stat().st_size for f in Path(CLIP_CACHE_DIR).glob("*.npy")) / 1e9
print(f"\nTotal cached: {n_cached:,} files  ({size_gb:.1f} GB)")
print("Set USE_CLIP_CACHE=True in config to use cache during training.")


Cache dir : /home/jinny/aspotting/clip_cache/e2_center
Clips to extract: 15,208



Train clips:   1%|          | 80/11702 [00:22<54:59,  3.52it/s]  


KeyboardInterrupt: 

In [ ]:
# 5c. Extract center-only clip cache (no jitter)
# Caches one clip per goal/negative at center_sec only (no ±2s offsets).
# Writes to a separate directory — existing jittered cache is untouched.

CENTER_CACHE_DIR = f"{PROJECT_DIR}/clip_cache/e2_center"
os.makedirs(CENTER_CACHE_DIR, exist_ok=True)

def center_cache_path(video_path, center_sec):
    key = f"{video_path}_{center_sec:.4f}"
    h   = hashlib.md5(key.encode()).hexdigest()
    return Path(CENTER_CACHE_DIR) / f"{h}.npy"

def extract_center_clips(df, desc="Extracting"):
    skipped = extracted = failed = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc):
        out = center_cache_path(row.video_path, float(row.center_sec))
        if out.exists():
            skipped += 1
            continue
        clip = read_clip(row.video_path, float(row.center_sec))
        if clip is None:
            failed += 1
            continue
        np.save(str(out), clip.numpy().astype(np.float16))
        extracted += 1
    print(f"  extracted={extracted}  skipped={skipped}  failed={failed}")

def build_center_only_samples(game_list):
    samples = []
    for game in tqdm(game_list, desc="Building center-only samples"):
        goal_times, hard_times = parse_annotations(game)
        for half in (1, 2):
            vid = Path(DATA_DIR, game, f"{half}_224p.mkv")
            if not vid.exists():
                continue
            cap   = cv2.VideoCapture(str(vid))
            fps_v = cap.get(cv2.CAP_PROP_FPS) or FPS
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
            dur = total / fps_v
            if dur < CLIP_SEC + 1:
                continue

            lo    = CLIP_SEC / 2
            hi    = dur - CLIP_SEC / 2
            goals = goal_times.get(half, [])
            hard  = hard_times.get(half, [])

            for g in goals:
                c = float(np.clip(g, lo, hi))
                samples.append((str(vid), c, 1.0, "goal"))

            if USE_HARD_NEGS:
                hard_cands = [(t, lbl) for t, lbl in hard
                              if far_from_all(t, goals, MIN_NEG_FROM_GOAL_SEC)]
                n_hard = min(len(hard_cands), NEG_HARD_PER_GOAL * max(1, len(goals)))
                if n_hard > 0:
                    for t, lbl in random.sample(hard_cands, k=n_hard):
                        samples.append((str(vid), float(np.clip(t, lo, hi)), 0.0, lbl))

            n_rand    = NEG_RAND_PER_GOAL * len(goals) if goals else NEG_PER_NO_GOAL_HALF
            forbidden = list(goals) + [t for t, _ in hard]
            added = attempts = 0
            while added < n_rand and attempts < n_rand * 40:
                attempts += 1
                t = random.uniform(lo, hi)
                if far_from_all(t, forbidden, MIN_NEG_FROM_GOAL_SEC):
                    samples.append((str(vid), t, 0.0, "rand"))
                    forbidden.append(t)
                    added += 1
    return samples

random.seed(42)
center_samples = build_center_only_samples(all_splits["train"] + all_splits["valid"])
center_df = pd.DataFrame(center_samples, columns=["video_path", "center_sec", "label", "neg_type"])

print(f"Center-only clips: {len(center_df):,}  "
      f"(pos={int((center_df.label==1.0).sum())}, neg={int((center_df.label==0.0).sum())})")
print(f"Cache dir: {CENTER_CACHE_DIR}")

extract_center_clips(center_df, desc="Center-only clips")

n_cached = len(list(Path(CENTER_CACHE_DIR).glob("*.npy")))
size_gb  = sum(f.stat().st_size for f in Path(CENTER_CACHE_DIR).glob("*.npy")) / 1e9
print(f"\nTotal cached: {n_cached:,} files  ({size_gb:.1f} GB)")

Building center-only samples: 100%|██████████| 400/400 [00:03<00:00, 112.71it/s]


Center-only clips: 10,334  (pos=1366, neg=8968)
Cache dir: /home/jinny/aspotting/clip_cache/e2_center


Center-only clips: 100%|██████████| 10334/10334 [57:21<00:00,  3.00it/s] 


  extracted=10331  skipped=3  failed=0

Total cached: 10,331 files  (12.4 GB)


In [7]:
# 6. Model
# R2Plus1D-18 pretrained on Kinetics-400.
# Single output neuron with BCEWithLogitsLoss for binary goal/no-goal.

model = r2plus1d_18(weights=R2Plus1D_18_Weights.KINETICS400_V1)

if FREEZE_BACKBONE:
    for name, param in model.named_parameters():
        if not (name.startswith("layer4") or name.startswith("fc")):
            param.requires_grad = False

model.fc = nn.Sequential(
    nn.Dropout(p=DROPOUT_P),
    nn.Linear(model.fc.in_features, 1)
)
model = model.to(DEVICE)

n_pos      = int((train_df.label == 1.0).sum())
n_neg      = int((train_df.label == 0.0).sum())
pos_weight = torch.tensor([n_neg / n_pos], device=DEVICE)
print(f"pos_weight: {pos_weight.item():.3f}")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler    = torch.amp.GradScaler(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Head      : {model.fc}")
print(f"Params    : {total:,}  (trainable: {trainable:,} / frozen: {total - trainable:,})")

pos_weight: 6.631
Head      : Sequential(
  (0): Dropout(p=0.4, inplace=False)
  (1): Linear(in_features=512, out_features=1, bias=True)
)
Params    : 31,300,638  (trainable: 31,300,638 / frozen: 0)


In [8]:
import os
os.environ["WANDB_ENTITY"] = "jintonyn"
print(os.environ["WANDB_ENTITY"])  # confirm it's right


jintonyn


In [9]:
# 7. Training loop

def run_epoch(loader, train):
    model.train() if train else model.eval()
    total_loss = correct = total = tp = fp = fn = 0
    all_preds, all_labels = [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for clips, labels in tqdm(loader, leave=False):
            clips  = clips.float().to(DEVICE)
            labels = labels.to(DEVICE)
            with torch.amp.autocast(DEVICE):
                logits = model(clips).squeeze(1)
                loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            preds       = (torch.sigmoid(logits) >= 0.5).float()
            total_loss += loss.item() * len(labels)
            correct    += (preds == labels).sum().item()
            total      += len(labels)
            tp += ((preds == 1) & (labels == 1)).sum().item()
            fp += ((preds == 1) & (labels == 0)).sum().item()
            fn += ((preds == 0) & (labels == 1)).sum().item()
            all_preds.extend(preds.int().cpu().tolist())
            all_labels.extend(labels.int().cpu().tolist())
    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return total_loss / total, correct / total, prec, rec, f1, all_preds, all_labels


def fp_breakdown_table(preds, labels, df):
    """
    Returns a wandb.Table of FP counts by neg_type.
    preds/labels are parallel lists matching df row order (shuffle=False).
    """
    fp_indices = [i for i, (p, l) in enumerate(zip(preds, labels)) if p == 1 and l == 0]
    counts = df["neg_type"].iloc[fp_indices].value_counts().to_dict()
    table = wandb.Table(columns=["neg_type", "fp_count"])
    for neg_type, count in sorted(counts.items(), key=lambda x: -x[1]):
        table.add_data(neg_type, count)
    return table


# ── Per-run checkpoint directory ──────────────────────────────────────────────
from datetime import datetime
run_dir     = Path(CKPT_DIR) / datetime.now().strftime("%Y%m%d_%H%M%S")
os.makedirs(run_dir, exist_ok=True)
best_ckpt   = str(run_dir / "best.pt")
latest_ckpt = str(run_dir / "latest.pt")
print(f"Checkpoint dir: {run_dir}")

# ── Resume from checkpoint ────────────────────────────────────────────────────
start_epoch   = 1
best_val_f1   = 0.0
wandb_run_id  = None
if RESUME_FROM:
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optim"])
    scheduler.load_state_dict(ckpt["sched"])
    start_epoch  = ckpt["epoch"] + 1
    best_val_f1  = ckpt.get("best_val_f1", 0.0)
    wandb_run_id = ckpt.get("wandb_run_id", None)
    print(f"Resumed from epoch {ckpt['epoch']}  best_val_f1={best_val_f1:.4f}  "
          f"wandb_run_id={wandb_run_id}")

# ── Training config summary ───────────────────────────────────────────────────
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("=" * 60)
print("TRAINING CONFIG")
print("=" * 60)
print(f"  Epochs        : {EPOCHS}  (start={start_epoch})")
print(f"  Batch size    : {BATCH_SIZE}  |  LR: {LR}  |  Weight decay: {WEIGHT_DECAY}")
print(f"  Clip          : {CLIP_SEC}s  {CLIP_FRAMES}f  {CLIP_SIZE[0]}px")
print(f"  Freeze backbone: {FREEZE_BACKBONE}  |  Dropout: {DROPOUT_P}")
print(f"  Augmentation  : {'ON (horizontal flip)' if USE_AUGMENTATION else 'OFF'}")
print(f"  Hard negatives: {USE_HARD_NEGS}  |  pos_weight: {pos_weight.item():.3f}")
print(f"  Params        : {trainable:,} trainable / {total:,} total")
print(f"  Train clips   : {len(train_df):,}  (pos={int((train_df.label==1.0).sum())}, neg={int((train_df.label==0.0).sum())})")
print(f"  Resume from   : {RESUME_FROM or '(none)'}")
print("=" * 60)

# ── W&B experiment ────────────────────────────────────────────────────────────
import os
run_name = (f"clip{CLIP_SEC}s_{'hardneg' if USE_HARD_NEGS else 'randneg'}_"
            f"lr{LR}_bs{BATCH_SIZE}_e{EPOCHS}")

run = wandb.init(
    project=os.environ["WANDB_PROJECT"],
    entity='jintonyn-oslomet',
    name=run_name,
    id=wandb_run_id,           # None on fresh start, run ID on resume
    resume="allow",            # continues existing run if id matches, creates new otherwise
    config={
        "clip_sec":             CLIP_SEC,
        "clip_frames":          CLIP_FRAMES,
        "clip_size":            CLIP_SIZE[0],
        "batch_size":           BATCH_SIZE,
        "epochs":               EPOCHS,
        "lr":                   LR,
        "weight_decay":         WEIGHT_DECAY,
        "freeze_backbone":      FREEZE_BACKBONE,
        "dropout_p":            DROPOUT_P,
        "use_augmentation":     USE_AUGMENTATION,
        "use_hard_negs":        USE_HARD_NEGS,
        "neg_hard_per_goal":    NEG_HARD_PER_GOAL,
        "neg_rand_per_goal":    NEG_RAND_PER_GOAL,
        "min_neg_from_goal_s":  MIN_NEG_FROM_GOAL_SEC,
        "stride_s":             STRIDE_S,
        "nms_radius_s":         NMS_RADIUS_S,
        "conf_thresh":          CONF_THRESH,
        "use_clip_cache":       USE_CLIP_CACHE,
        "n_train_clips":        len(train_df),
        "n_valid_clips":        len(valid_df),
        "n_train_pos":          int((train_df.label == 1.0).sum()),
        "n_train_neg":          int((train_df.label == 0.0).sum()),
        "pos_weight":           float(pos_weight.item()),
        "device":               DEVICE,
        "resume_from":          RESUME_FROM or None,
        "start_epoch":          start_epoch,
    }
)

# Log manifests as artifacts
artifact = wandb.Artifact(name="manifests", type="dataset")
artifact.add_file(train_csv, name="train_manifest.csv")
artifact.add_file(valid_csv, name="valid_manifest.csv")
run.log_artifact(artifact)
print("W&B run started:", run.name, " id:", run.id)

# ── Training ──────────────────────────────────────────────────────────────────
for epoch in range(start_epoch, EPOCHS + 1):
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(valid_loader, train=False)
    scheduler.step()

    print(f"Epoch {epoch:02d}/{EPOCHS}  "
          f"train loss={tr[0]:.4f} acc={tr[1]:.3f} F1={tr[4]:.3f}  |  "
          f"val   loss={va[0]:.4f} acc={va[1]:.3f} F1={va[4]:.3f} "
          f"P={va[2]:.3f} R={va[3]:.3f}")

    wandb.log({
        "train/loss": tr[0], "train/acc": tr[1],
        "train/prec": tr[2], "train/rec": tr[3], "train/f1": tr[4],
        "val/loss":   va[0], "val/acc":   va[1],
        "val/prec":   va[2], "val/rec":   va[3], "val/f1":   va[4],
    }, step=epoch)
    wandb.log({
        "val/confusion_matrix": wandb.plot.confusion_matrix(
            y_true=va[6], preds=va[5],
            class_names=["background", "goal"],
        ),
        "val/fp_by_neg_type": fp_breakdown_table(va[5], va[6], valid_df),
    }, step=epoch)

    torch.save({"epoch": epoch, "model": model.state_dict(),
                "optim": optimizer.state_dict(),
                "sched": scheduler.state_dict(),
                "val_f1": va[4],
                "best_val_f1": best_val_f1,
                "wandb_run_id": run.id}, latest_ckpt)

    if va[4] > best_val_f1:
        best_val_f1 = va[4]
        torch.save(model.state_dict(), best_ckpt)
        wandb.log({"best_val_f1": best_val_f1}, step=epoch)
        wandb.save(best_ckpt)
        print(f"  -> saved best checkpoint  (val F1={best_val_f1:.4f})")

run.finish()
print("Training complete.")
print(f"Checkpoints saved to: {run_dir}")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


Checkpoint dir: /home/jinny/aspotting/checkpoints/e2/20260323_123732
TRAINING CONFIG
  Epochs        : 8  (start=1)
  Batch size    : 16  |  LR: 0.0001  |  Weight decay: 0.001
  Clip          : 4.0s  16f  112px
  Freeze backbone: False  |  Dropout: 0.4
  Augmentation  : ON (horizontal flip)
  Hard negatives: True  |  pos_weight: 6.631
  Params        : 31,300,638 trainable / 31,300,638 total
  Train clips   : 7,593  (pos=995, neg=6598)
  Resume from   : (none)


wandb: Currently logged in as: jintonyn to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run started: clip4.0s_hardneg_lr0.0001_bs16_e8  id: 5wvc2jdo


  4%|▍         | 19/474 [00:08<02:53,  2.62it/s]wandb: WARNING Artifact "manifests" already exists with the same content. No new version will be created.


...

Epoch 08/8  train loss=0.0254 acc=0.996 F1=0.983  |  val   loss=0.3741 acc=0.963 F1=0.868 P=0.834 R=0.906


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


  -> saved best checkpoint  (val F1=0.8682)


best_val_f1,▁▃▅▇▇██
train/acc,▁▄▅▆▇▇██
train/f1,▁▃▅▅▆▇██
train/loss,█▅▄▃▂▂▁▁
train/prec,▁▃▄▅▆▇██
train/rec,▁▄▅▆▆▇██
val/acc,▁▃▅▂▇▇██
val/f1,▁▃▅▂▇▇██
val/loss,▆▃▁█▃▄▅▆
val/prec,▁▂▅▂▆███
+1,...


Training complete.
Checkpoints saved to: /home/jinny/aspotting/checkpoints/e2/20260323_123732


In [ ]:
# 8. Sliding-window inference

def _moving_average(arr, k):
    if k <= 1:
        return arr
    pad    = k // 2
    padded = np.pad(arr, (pad, pad), mode='edge')
    return np.convolve(padded, np.ones(k, dtype=np.float32) / k, mode='valid')


def sliding_window_inference(video_path, conf_thresh=CONF_THRESH,
                              nms_radius_s=NMS_RADIUS_S, smooth_k=1):
    """
    Slides a window over the half and returns (detections, raw_curve).
    conf_thresh  : minimum confidence to keep a detection
    nms_radius_s : suppression radius in seconds
    smooth_k     : moving-average window over the confidence curve (1 = off)

    raw_curve is a list of (center_sec, prob) before smoothing/NMS — save it
    to re-run post-processing without re-running inference.
    """
    cap          = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_v        = cap.get(cv2.CAP_PROP_FPS) or FPS
    cap.release()

    dur       = total_frames / fps_v
    centers_s = np.arange(CLIP_SEC / 2, dur - CLIP_SEC / 2, STRIDE_S)

    model.eval()
    raw            = []   # (center_sec, goal_prob)
    batch_clips    = []
    batch_centers  = []

    def _flush():
        if not batch_clips:
            return
        t = torch.stack(batch_clips).float().to(DEVICE)
        with torch.no_grad(), torch.amp.autocast(DEVICE):
            probs = torch.sigmoid(model(t).squeeze(1)).cpu().numpy()
        for cs, p in zip(batch_centers, probs):
            raw.append((float(cs), float(p)))
        batch_clips.clear()
        batch_centers.clear()

    for cs in tqdm(centers_s, desc=Path(video_path).name, leave=False):
        clip = read_clip(video_path, float(cs))
        if clip is None:
            continue
        batch_clips.append(clip)
        batch_centers.append(cs)
        if len(batch_clips) == BATCH_SIZE:
            _flush()
    _flush()

    if not raw:
        return [], []

    raw_curve = sorted(raw, key=lambda x: x[0])   # chronological, unsmoothed
    detections = postprocess(raw_curve, conf_thresh, nms_radius_s, smooth_k)
    return detections, raw_curve


def postprocess(raw_curve, conf_thresh=CONF_THRESH,
                nms_radius_s=NMS_RADIUS_S, smooth_k=1):
    """
    Apply smoothing + threshold + NMS to a raw confidence curve.
    raw_curve is a list of (center_sec, prob) in chronological order.
    Returns detections sorted by timestamp.
    """
    if not raw_curve:
        return []
    smoothed = raw_curve
    if smooth_k > 1:
        cs_arr   = [x[0] for x in raw_curve]
        p_arr    = _moving_average(np.array([x[1] for x in raw_curve], np.float32), smooth_k)
        smoothed = list(zip(cs_arr, p_arr.tolist()))
    candidates = sorted(smoothed, key=lambda x: -x[1])
    detections = []
    for cs, prob in candidates:
        if prob < conf_thresh:
            continue
        if all(abs(cs - d["timestamp_s"]) >= nms_radius_s for d in detections):
            detections.append({"timestamp_s": cs, "confidence": prob})
    detections.sort(key=lambda x: x["timestamp_s"])
    return detections


print("Sliding-window inference defined")

Sliding-window inference defined


In [ ]:
# 7b. Single-half sanity check
# Scores one validation half and prints detected goals vs ground truth.
# Adjust the variables below without touching the main config.

SANITY_THRESH   = 0.9   # lower = more detections, higher = fewer but more confident
SANITY_SMOOTH_K = 7     # moving-average window over confidence curve (1 = off, try 5 or 7)
SANITY_NMS_S    = 10.0  # NMS suppression radius in seconds
SANITY_TOL_S    = 5.0   # tolerance for HIT/FP labelling

sanity_game = all_splits["valid"][2]
sanity_half = 1
sanity_vid  = Path(DATA_DIR, sanity_game, f"{sanity_half}_224p.mkv")
gt_goals, _ = parse_annotations(sanity_game)
gt_goals    = gt_goals.get(sanity_half, [])
best_ckpt = "/home/jinny/aspotting/checkpoints/e2/20260317_082355/best.pt"

print(f"Game : {sanity_game}")
print(f"Half : {sanity_half}")
print(f"GT goals (s): {[round(t, 1) for t in gt_goals]}")
print(f"Threshold={SANITY_THRESH}  smooth_k={SANITY_SMOOTH_K}  nms={SANITY_NMS_S}s\n")

model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()

dets, _ = sliding_window_inference(
    str(sanity_vid),
    conf_thresh=SANITY_THRESH,
    nms_radius_s=SANITY_NMS_S,
    smooth_k=SANITY_SMOOTH_K,
)

print(f"Detected {len(dets)} event(s):")
for d in dets:
    t = d['timestamp_s']
    mm, ss  = int(t) // 60, int(t) % 60
    matched = any(abs(t - gt) <= SANITY_TOL_S for gt in gt_goals)
    print(f"  {mm:02d}:{ss:02d}  conf={d['confidence']:.3f}  {'HIT' if matched else 'FP'}")

if not dets:
    print("  (no detections above threshold)")

missed = [gt for gt in gt_goals
          if not any(abs(gt - d['timestamp_s']) <= SANITY_TOL_S for d in dets)]
tp = len(gt_goals) - len(missed)
fp = sum(1 for d in dets
         if not any(abs(d['timestamp_s'] - gt) <= SANITY_TOL_S for gt in gt_goals))
print(f"\nMissed goals : {[round(t, 1) for t in missed]}")
print(f"TP={tp}  FP={fp}  FN={len(missed)}")

Game : england_epl/2015-2016/2015-09-26 - 17-00 Leicester 2 - 5 Arsenal
Half : 1
GT goals (s): [731.5, 1052.8, 1972.9]
Threshold=0.9  smooth_k=7  nms=10.0s



RuntimeError: Error(s) in loading state_dict for VideoResNet:
	Missing key(s) in state_dict: "fc.1.weight", "fc.1.bias". 
	Unexpected key(s) in state_dict: "fc.weight", "fc.bias". 

In [ ]:
# 9. Run inference on the validation set
import pickle

RAW_CURVES_PATH = f"{PROJECT_DIR}/inference_raw_e2.pkl"

model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.eval()
print(f"Loaded: {best_ckpt}")

all_preds = {}   # (game, half) -> [(timestamp_s, confidence), ...]
all_gt    = {}   # (game, half) -> [timestamp_s, ...]
all_raw   = {}   # (game, half) -> [(center_sec, prob), ...]  — unsmoothed, pre-NMS

for game in tqdm(all_splits["valid"], desc="Inference"):
    goal_times, _ = parse_annotations(game)
    for half in (1, 2):
        vid = Path(DATA_DIR, game, f"{half}_224p.mkv")
        if not vid.exists():
            continue
        key            = (game, half)
        all_gt[key]    = goal_times.get(half, [])
        dets, raw_curve = sliding_window_inference(str(vid))
        all_preds[key] = [(d["timestamp_s"], d["confidence"]) for d in dets]
        all_raw[key]   = raw_curve

with open(RAW_CURVES_PATH, "wb") as f:
    pickle.dump({"all_raw": all_raw, "all_gt": all_gt}, f)

print(f"Done. Halves processed: {len(all_preds)}")
print(f"Raw curves saved to: {RAW_CURVES_PATH}")

Loaded: /home/jinny/aspotting/checkpoints/e2/20260317_082355/best.pt


Inference:   0%|          | 0/100 [03:06<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
# 9b. Re-run post-processing on saved raw curves (no inference needed)
# Tweak conf_thresh, smooth_k, nms_radius_s freely here.
import pickle

RAW_CURVES_PATH = f"{PROJECT_DIR}/inference_raw_e2.pkl"

PP_THRESH   = 0.5
PP_SMOOTH_K = 7
PP_NMS_S    = 10.0

with open(RAW_CURVES_PATH, "rb") as f:
    saved = pickle.load(f)

all_raw = saved["all_raw"]
all_gt  = saved["all_gt"]

all_preds = {
    key: [(d["timestamp_s"], d["confidence"])
          for d in postprocess(curve, PP_THRESH, PP_NMS_S, PP_SMOOTH_K)]
    for key, curve in all_raw.items()
}

print(f"Re-scored {len(all_preds)} halves  "
      f"(thresh={PP_THRESH}  smooth_k={PP_SMOOTH_K}  nms={PP_NMS_S}s)")

In [ ]:
# 10. Evaluation -- Average mAP at multiple tolerances
# Mirrors the SoccerNet action spotting benchmark:
# a detection is TP if it is within tol seconds of the nearest unmatched GT goal.

def compute_ap(preds_with_conf, gt_timestamps, tol):
    preds   = sorted(preds_with_conf, key=lambda x: -x[1])
    matched = set()
    tp_list, fp_list = [], []
    for pred_t, _ in preds:
        best, best_d = None, tol
        for i, gt_t in enumerate(gt_timestamps):
            if i not in matched and abs(pred_t - gt_t) <= best_d:
                best_d = abs(pred_t - gt_t)
                best   = i
        if best is not None:
            matched.add(best); tp_list.append(1); fp_list.append(0)
        else:
            tp_list.append(0); fp_list.append(1)
    if not tp_list or not gt_timestamps:
        return 0.0
    tp_cum = np.cumsum(tp_list)
    fp_cum = np.cumsum(fp_list)
    prec   = tp_cum / (tp_cum + fp_cum + 1e-8)
    rec    = tp_cum / len(gt_timestamps)
    ap, prev_r = 0.0, 0.0
    for p, r in zip(prec, rec):
        ap += p * (r - prev_r); prev_r = r
    return ap


print(f"{'Tolerance':>10s}  {'mAP':>8s}")
print("-" * 22)
for tol in TOLERANCES:
    aps = [compute_ap(all_preds.get(k, []), all_gt[k], tol)
           for k in all_gt if all_gt[k]]
    print(f"{tol:>9d}s  {np.mean(aps) * 100:>7.2f}%" if aps else f"{tol:>9d}s  N/A")

In [ ]:
# 11. Single-game demo on test split
import random

demo_game = random.choice(all_splits["test"])
demo_half = 1
demo_vid  = Path(DATA_DIR, demo_game, f"{demo_half}_224p.mkv")

print(f"Game : {demo_game}")
gt_goals, _ = parse_annotations(demo_game)
gt_goals    = gt_goals.get(demo_half, [])
print(f"GT goals (s): {[f'{t:.1f}' for t in sorted(gt_goals)]}")

if demo_vid.exists():
    dets = sliding_window_inference(str(demo_vid))
    print("\nPredicted goal timestamps:")
    for d in dets:
        m = int(d['timestamp_s'] // 60)
        s = int(d['timestamp_s'] % 60)
        print(f"  {m:02d}:{s:02d}  conf={d['confidence']:.3f}")
else:
    print("Video not found locally.")
